# Источники трафика и категории

Анализ открытий карточек и заказов за 30 дней. Источники описывают открытия, а не заказы. Состав и точная атрибуция источников могут различаться между таблицами.

## Как читать маршруты: уточнение для анализа

Ниже используется **рабочая трактовка, уточнённая командой**, а не доказанное по выгрузке техническое определение каждого события:

- **Маркет VK:** открытие карточки товара, относящегося к Маркету, во VK → нажатие кнопки → возможный переход к чекауту VK или в чат с продавцом. Кейс упоминает и внешние сайты; фактические назначения кнопок в выгрузке не разделены.
- **Партнёр 1 через карточку VK:** открытие карточки его товара во VK → нажатие кнопки → переход к чекауту на сайте партнёра (по уточнённому сценарию). Это ещё не оформленный заказ.
- **Партнёр 1 напрямую:** переход из VK без открытия карточки во VK → открытие страницы товара на сайте партнёра (по уточнённому сценарию); в данных этот маршрут отражён как прямые переходы, а не как открытия карточки VK.

Партнёр 1 — условная площадка, не обязательно Ozon. Товар Маркета может продаваться и на других площадках. Назначения кнопок и факт открытия страницы партнёра нужно подтвердить по справочнику событий и интеграции; эти детали отдельно в CSV не измерены. Прямые переходы и нажатия ведут на разные шаги и могут пересекаться по пользователям: их сумма не является числом уникальных покупателей или единым знаменателем конверсии. Сравнение агрегированных отношений — не когортная конверсия и не диагностика причин.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

data_path = Path.cwd().parent / 'Data'
mv = 'Маркет Вконтакте'
p1 = 'Партнёр 1'
pd.set_option('display.float_format', '{:,.2f}'.format)

traffic = pd.read_csv(data_path / 'Market_Partner_traffic.csv').set_index('Источник')
categories = pd.read_csv(data_path / 'Market_Partner_categories.csv').set_index('Категория')
market = pd.read_csv(data_path / 'Market_Partner.csv').set_index('Метрика')
total = pd.read_csv(data_path / 'Market_Total.csv').set_index('Период')
traffic

,Маркет Вконтакте,"Доля, %: Маркет Вконтакте",Партнёр 1,"Доля, %: Партнёр 1"
Источник,,,,
Другое,"77,753,037.00",40.20,5380616,6.20
Витрина магазина,"75,604,323.00",39.10,714540,0.80
Лента,"18,795,696.00",9.70,15622386,18.00
Клипы,"17,239,066.00",8.90,64917109,74.70
Поиск,"3,707,521.00",1.90,159788,0.20
Главная Маркета Вконтакте,"412,818.00",0.20,88383,0.10
в т.ч. из реферальных постов и клипов,NaN,NaN,82324081,94.80


In [ ]:
# Интерактивные визуализации
import plotly.graph_objects as go

def show_plotly(fig, title, x_title, y_title, height=500):
    fig.update_layout(
        title=dict(text=title), template='plotly_white', height=height,
        xaxis_title=x_title, yaxis_title=y_title,
        legend_title_text='Канал', margin=dict(l=70, r=30, t=80, b=90),
        font=dict(size=13), hoverlabel=dict(namelength=-1),
    )
    fig.show(config={'displaylogo': False, 'responsive': True})


## Источники открытий

Строка «в т.ч.» — вложенное сечение, а не ещё один независимый источник; она исключена из суммы.
Это источники открытий карточек во VK. Прямые переходы партнёра на внешний сайт без карточки VK в структуру таких открытий не входят.


In [2]:
sources = traffic.loc[~traffic.index.str.startswith('в т.ч.')].copy()
for channel in [mv, p1]:
    sources[f'Доля по пересчёту, %: {channel}'] = sources[channel] / sources[channel].sum() * 100
sources

,Маркет Вконтакте,"Доля, %: Маркет Вконтакте",Партнёр 1,"Доля, %: Партнёр 1","Доля по пересчёту, %: Маркет Вконтакте","Доля по пересчёту, %: Партнёр 1"
Источник,,,,,,
Другое,"77,753,037.00",40.20,5380616,6.20,40.18,6.19
Витрина магазина,"75,604,323.00",39.10,714540,0.80,39.07,0.82
Лента,"18,795,696.00",9.70,15622386,18.00,9.71,17.98
Клипы,"17,239,066.00",8.90,64917109,74.70,8.91,74.72
Поиск,"3,707,521.00",1.90,159788,0.20,1.92,0.18
Главная Маркета Вконтакте,"412,818.00",0.20,88383,0.10,0.21,0.10


### Интерактивно: структура источников открытий
Доля рассчитана отдельно внутри суммы источников каждого канала. Вложенная строка «в т.ч.» исключена; данные не содержат заказов по источникам.


In [ ]:
fig = go.Figure()
for channel, color in [(mv, '#3978C6'), (p1, '#F2994A')]:
    fig.add_bar(name=channel, x=sources[f'Доля по пересчёту, %: {channel}'], y=sources.index,
                orientation='h', marker_color=color, customdata=sources[channel],
                hovertemplate='%{y}<br>%{fullData.name}: %{x:.2f}%<br>Открытий: %{customdata:,.0f}<extra></extra>')
fig.update_layout(barmode='group')
fig.update_xaxes(rangemode='tozero', ticksuffix='%')
fig.update_yaxes(autorange='reversed')
show_plotly(fig, 'Источники открытий по каналам', 'Доля открытий в сумме источников канала, %', '', 560)


Интерактивный график Plotly: Источники открытий по каналам

In [3]:
traffic_check = pd.DataFrame({'Сумма источников': sources[[mv, p1]].sum(), 'Открытия в итогах': market.loc['Открытия карточки товара', [mv, p1]]})
traffic_check['Разница'] = traffic_check['Сумма источников'] - traffic_check['Открытия в итогах']
traffic_check

,Сумма источников,Открытия в итогах,Разница
Маркет Вконтакте,"193,512,461.00","193,512,461.00",0.00
Партнёр 1,"86,882,822.00","86,882,822.00",0.00


In [4]:
pd.Series({'Лента + Клипы партнёра': sources.loc[['Лента', 'Клипы'], p1].sum(), 'Реферальные открытия партнёра': traffic.loc['в т.ч. из реферальных постов и клипов', p1], 'Лента + Клипы обоих каналов': sources.loc[['Лента', 'Клипы'], [mv, p1]].to_numpy().sum(), 'Посты и клипы по общей таблице': total.loc['Открытия карточки товара из постов и клипов', 'Значение'], 'Главная Маркета / все открытия источников, %': sources.loc['Главная Маркета Вконтакте', [mv, p1]].sum() / sources[[mv, p1]].to_numpy().sum() * 100})

Лента + Клипы партнёра                          80,539,495.00
Реферальные открытия партнёра                   82,324,081.00
Лента + Клипы обоих каналов                    116,574,257.00
Посты и клипы по общей таблице                  86,321,893.00
Главная Маркета / все открытия источников, %             0.18
dtype: float64

У Маркета 39,1% открытий приходится на витрины магазинов и 40,2% на «Другое»; у партнёра 74,7% — на клипы. Главная Маркета обеспечивает около 0,18% открытий двух каналов. Реферальные открытия партнёра превышают сумму его ленты и клипов, а общая таблица также не сходится с суммой этих источников двух каналов. Определения и атрибуцию нужно уточнить; объединять такие показатели в единую воронку нельзя. Нет заказов по источникам, поэтому эффективность источника по продажам неизвестна.

## Категории и качество покрытия

Проверим итоги: категории могут пересекаться или иметь несовпадающую атрибуцию. Доли категорий ниже нормированы на сумму категорий каждого канала, а не на общий итог.

In [5]:
category_check = pd.DataFrame(index=[mv, p1])
for channel in [mv, p1]:
    category_check.loc[channel, 'Открытия: категории'] = categories[f'Открытия карточки: {channel}'].sum()
    category_check.loc[channel, 'Открытия: итог'] = market.loc['Открытия карточки товара', channel]
    category_check.loc[channel, 'Заказы: категории'] = categories[f'Заказы: {channel}'].sum()
    category_check.loc[channel, 'Заказы: итог'] = market.loc['Оформленные заказы', channel]
category_check

,Открытия: категории,Открытия: итог,Заказы: категории,Заказы: итог
Маркет Вконтакте,"193,472,512.00","193,512,461.00","435,285.00","413,740.00"
Партнёр 1,"86,880,602.00","86,882,822.00","4,557,517.00","4,927,525.00"


In [6]:
category_metrics = categories.copy()
for channel in [mv, p1]:
    opens = categories[f'Открытия карточки: {channel}']
    orders = categories[f'Заказы: {channel}']
    category_metrics[f'Доля открытий, %: {channel}'] = opens / opens.sum() * 100
    category_metrics[f'Доля заказов, %: {channel}'] = orders / orders.sum() * 100
    category_metrics[f'Заказы / открытия, %: {channel}'] = orders / opens.replace(0, np.nan) * 100
category_metrics['Доля Маркета в заказах категории, %'] = categories[f'Заказы: {mv}'] / (categories[f'Заказы: {mv}'] + categories[f'Заказы: {p1}']).replace(0, np.nan) * 100
category_metrics.sort_values(f'Заказы: {mv}', ascending=False)

,Открытия карточки: Маркет Вконтакте,Заказы: Маркет Вконтакте,Открытия карточки: Партнёр 1,Заказы: Партнёр 1,"Доля открытий, %: Маркет Вконтакте","Доля заказов, %: Маркет Вконтакте","Заказы / открытия, %: Маркет Вконтакте","Доля открытий, %: Партнёр 1","Доля заказов, %: Партнёр 1","Заказы / открытия, %: Партнёр 1","Доля Маркета в заказах категории, %"
Категория,,,,,,,,,,,
Продукты питания,20393319,195355,2488477,170556,10.54,44.88,0.96,2.86,3.74,6.85,53.39
Хобби и развлечения,32048705,110703,2474633,116999,16.56,25.43,0.35,2.85,2.57,4.73,48.62
Дом и дача,26162824,35976,24528887,1413965,13.52,8.26,0.14,28.23,31.02,5.76,2.48
Гардероб,33022954,29029,9023244,259808,17.07,6.67,0.09,10.39,5.70,2.88,10.05
Красота и здоровье,11345073,13022,14113274,1240324,5.86,2.99,0.11,16.24,27.21,8.79,1.04
Детские товары,4859516,10546,2478504,102803,2.51,2.42,0.22,2.85,2.26,4.15,9.30
Товары для животных,2167931,8600,1793132,146558,1.12,1.98,0.40,2.06,3.22,8.17,5.54
Транспорт,8590002,8254,4569160,264510,4.44,1.90,0.10,5.26,5.80,5.79,3.03
Спорт и отдых,5632947,7781,4092740,179554,2.91,1.79,0.14,4.71,3.94,4.39,4.15


### Интерактивно: заказы Маркета по категориям
Показаны все категории, включая «Другое» и агрегированные категории. Они могут пересекаться; сумма строк не равна общему итогу заказов.


In [ ]:
cat_orders = categories[f'Заказы: {mv}'].sort_values()
fig = go.Figure(go.Bar(x=cat_orders.values, y=cat_orders.index, orientation='h', marker_color='#3978C6',
                       hovertemplate='%{y}<br>Заказы Маркета: %{x:,.0f}<extra></extra>'))
fig.update_xaxes(rangemode='tozero')
show_plotly(fig, 'Оформленные заказы Маркета по категориям', 'Заказы, шт.', '', 740)


Интерактивный график Plotly: Оформленные заказы Маркета по категориям

In [7]:
category_metrics[[f'Заказы / открытия, %: {mv}', f'Заказы / открытия, %: {p1}']].corr(method='spearman')

,"Заказы / открытия, %: Маркет Вконтакте","Заказы / открытия, %: Партнёр 1"
"Заказы / открытия, %: Маркет Вконтакте",1.00,0.70
"Заказы / открытия, %: Партнёр 1",0.70,1.00


Продукты питания и хобби дают большую часть учтённых заказов Маркета; по отношению к сумме двух каналов в этих категориях их доля около 53% и 49%. Это перспективные направления для проверки предложения локальным продавцам, но данные не устанавливают размер бизнеса и прибыль. Отношение заказов к открытиям партнёра не является чистой конверсией карточки из-за прямых переходов. Нулевые или очень малые знаменатели создают нестабильные отношения. Итоги заказов по категориям расходятся с общими итогами: категории нельзя автоматически суммировать в непротиворечивую воронку.

## Сценарный расчёт потенциала

Не прогноз: фиксируем открытия и повышаем отношение заказов к открытиям на 0,1 процентного пункта в пяти содержательно определённых категориях с наибольшим трафиком Маркета. «Другое», «Все товары» и «Не определена» исключены. Не учитываем ограничение предложения, пересечения категорий и изменение поведения.
Это изменение агрегированного отношения, а не доказанная конверсия карточки или оценка новых продаж. Маршруты чата, чекаута и внешней покупки не выделены.


In [8]:
scenario = categories.drop(index=['Другое', 'Все товары', 'Не определена']).sort_values(f'Открытия карточки: {mv}', ascending=False).head(5)[[f'Открытия карточки: {mv}', f'Заказы: {mv}']].copy()
scenario['Сценарно дополнительные заказы при +0,1 п.п.'] = scenario[f'Открытия карточки: {mv}'] * 0.001
scenario

,Открытия карточки: Маркет Вконтакте,Заказы: Маркет Вконтакте,"Сценарно дополнительные заказы при +0,1 п.п."
Категория,,,
Гардероб,33022954,29029,"33,022.95"
Хобби и развлечения,32048705,110703,"32,048.71"
Дом и дача,26162824,35976,"26,162.82"
Продукты питания,20393319,195355,"20,393.32"
Красота и здоровье,11345073,13022,"11,345.07"


### Интерактивно: условный сценарий +0,1 п.п.
Расчёт — открытия × 0,001, при фиксированных открытиях. Это не прогноз; категории могут пересекаться.


In [ ]:
scenario_plot = scenario['Сценарно дополнительные заказы при +0,1 п.п.'].sort_values()
fig = go.Figure(go.Bar(x=scenario_plot.values, y=scenario_plot.index, orientation='h', marker_color='#3978C6',
                       hovertemplate='%{y}<br>Условно дополнительных заказов: %{x:,.1f}<extra></extra>'))
fig.update_xaxes(rangemode='tozero')
show_plotly(fig, 'Условный эффект +0,1 п.п. в пяти категориях', 'Расчётное число дополнительных заказов, шт.', '', 450)


Интерактивный график Plotly: Условный эффект +0,1 п.п. в пяти категориях

In [9]:
pd.Series({'Сумма сценарных дополнительных заказов': scenario['Сценарно дополнительные заказы при +0,1 п.п.'].sum(), 'Относительно итоговых заказов Маркета, %': scenario['Сценарно дополнительные заказы при +0,1 п.п.'].sum() / market.loc['Оформленные заказы', mv] * 100})

Сумма сценарных дополнительных заказов     122,972.88
Относительно итоговых заказов Маркета, %        29.72
dtype: float64

Сценарий показывает порядок величины, но из-за несогласованности категорий и общего итога его нельзя считать прогнозом роста. В данных нет GMV по категориям и рекламных расходов, поэтому нельзя оценить выручку или окупаемость категории.